In [1]:
import pandas as pd
from pathlib import Path

In [2]:
repo = Path(".").resolve().parent
json_path = repo / "data/raw/json"
print(json_path)

/home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/json


In [3]:
from Python_code.modules.json_parser_theo import load_chats_from_folder
df = load_chats_from_folder(json_path)
len(df.groupby("chat_id"))

30

In [4]:
df

,chat_id,file_name,turn,role,content
0,1,chat_export,1,user,wie codiere ich alle values in einen pd df zu ...
1,1,chat_export,2,assistant,In pandas kannst du alle negativen Werte im Da...
2,1,chat_export,3,user,Exportiere mir den chat bis hierher als json D...
3,2,copilot,1,user,Was sind wichtige Werke in der Literatur zur r...
4,2,copilot,2,assistant,**Kurzfazit:** \nDie Literatur zur **Reliabil...
...,...,...,...,...,...
274,29,role user content was,2,assistant,Mögliche Motivationen für die Replikation der ...
275,30,role user content was2,1,user,was ist der unteschied zwischen computer visio...
276,30,role user content was2,2,assistant,"Der zentrale Unterschied liegt darin, **wie Mo..."
277,30,role user content was2,3,user,i need literatur on how to evaluate zero shot ...


In [5]:

anzahl_dateien = len([f for f in json_path.iterdir() if f.is_file()])

print(anzahl_dateien)

30


In [6]:
#df.groupby("chat_id")["turn"].max().eq(1)

In [7]:
df_user = df[df["role"] == "user"]
df_user

,chat_id,file_name,turn,role,content
0,1,chat_export,1,user,wie codiere ich alle values in einen pd df zu ...
2,1,chat_export,3,user,Exportiere mir den chat bis hierher als json D...
3,2,copilot,1,user,Was sind wichtige Werke in der Literatur zur r...
6,2,copilot,4,user,Exportiere den gesamten im aktuellen Kontext e...
8,2,copilot,6,user,Exportiere den gesamten sichtbaren Chatverlauf...
...,...,...,...,...,...
269,27,role user content erkläre,1,user,erkläre mir diese formel in bezug auf neuronal...
271,28,role user content regression,1,user,regression with categorical outcome an panel data
273,29,role user content was,1,user,was wären motivationen für die replikation die...
275,30,role user content was2,1,user,was ist der unteschied zwischen computer visio...


In [8]:
import numpy as np
#shuffel dataset and reset ids
rng = np.random.default_rng(seed=42)
unique_ids = df_user['chat_id'].unique()
shuffled_ids = rng.permutation(unique_ids)

id_mapping = {old_id: new_id for new_id, old_id in enumerate(shuffled_ids, start=1)}

df_shuffled = (
    pd.concat([df_user[df_user['chat_id'] == i] for i in shuffled_ids])
    .reset_index(drop=True)
)

df_shuffled['chat_id'] = df_shuffled['chat_id'].map(id_mapping)

df_shuffled


,chat_id,file_name,turn,role,content
0,1,role user content was2,1,user,was ist der unteschied zwischen computer visio...
1,1,role user content was2,3,user,i need literatur on how to evaluate zero shot ...
2,2,laura-06,1,user,Version control conflict markers detected. Ple...
3,2,laura-06,2,user,what if I don't want to keep any of my changes...
4,2,laura-06,3,user,now my old version is the latest version and t...
...,...,...,...,...,...
156,30,gemini7,6,user,Exportiere den gesamten im aktuellen Kontext e...
157,30,gemini7,7,user,Schreibe einen Text für meine Hausarbeit (Repl...
158,30,gemini7,9,user,Alles in einem Fließtext ohne Überschriften bi...
159,30,gemini7,11,user,Wie nennt man es wenn man einer studie komplet...


In [9]:

df_control = pd.DataFrame({
    "chat_id": range(1, 31),
    "task": ""*30,
    "sentiment": ""*30,
    "critical": ""*30
})

In [10]:
df_control

,chat_id,task,sentiment,critical
0,1,,,
1,2,,,
2,3,,,
3,4,,,
4,5,,,
5,6,,,
6,7,,,
7,8,,,
8,9,,,
9,10,,,


In [11]:
df_shuffled.to_csv(repo / "data/processed/control/user_shuffel.csv", index=False)
df_control.to_csv(repo / "data/processed/control/control.csv", index=False)
df_user.to_csv(repo / "data/processed/control/user.csv", index=False)

In [12]:
#!pip install krippendorff

In [13]:

import pandas as pd
from sklearn.metrics import cohen_kappa_score
import krippendorff

In [14]:
control_a_task = pd.read_csv(repo / "data/processed/control/control_hannah.csv", delimiter = ";")
control_b_task = pd.read_csv(repo / "data/processed/control/control_theo_2.csv")
control_c_task = pd.read_csv(repo / "data/processed/control/control_laura.csv")

control_a = pd.read_csv(repo / "data/processed/control/control_hannah_2.csv", delimiter= ";")
control_b = pd.read_csv(repo / "data/processed/control/round_2.csv")
control_c = pd.read_csv(repo / "data/processed/control/control_laura_runde2.csv")



In [15]:
def kippendorff_alpha(name: str,
                      annontation = ["code_coder_A", "code_coder_C",  "code_coder_B"],
                      messumerment = "nominal"
                        ):
    coder_a = control_a[["chat_id", name]].rename(columns={name: "code_coder_A"})
    coder_b = control_b[["chat_id", name]].rename(columns={name: "code_coder_B"})
    coder_c = control_c[["chat_id", name]].rename(columns={name: "code_coder_C"})
    merged = coder_a.merge(coder_b, on="chat_id").merge(coder_c, on="chat_id")#
    alpha = krippendorff.alpha(
    reliability_data=merged[annontation].T.to_numpy(),
    level_of_measurement=messumerment
    )
    not_matched = merged[merged[annontation].nunique(axis=1) > 1]
    return alpha, not_matched["chat_id"].tolist()


In [16]:
kippendorff_alpha(name = "sentiment", annontation = ["code_coder_A", "code_coder_C",  "code_coder_B"], messumerment = "ordinal")

(np.float64(0.4597943069063759), [7, 13, 25, 29])

In [17]:
coderABC = ["code_coder_A", "code_coder_C", "code_coder_B"]
coder_AB = ["code_coder_A", "code_coder_B"]
coder_BC = ["code_coder_B", "code_coder_C"]
coder_AC = ["code_coder_A", "code_coder_C"]


for coder in  [coderABC,coder_AB, coder_BC,coder_AC]:
    print("")
    for name, messure in zip(["sentiment", "critical"], ["ordinal", "nominal"]):
        alpha, no = kippendorff_alpha(name, annontation = coder, messumerment = messure)
        print(f"Annotators : {coder};" f"Krippendorff's alpha for {name}: {alpha:.4f}")
        print(f"chat_id with no match: {no}")


Annotators : ['code_coder_A', 'code_coder_C', 'code_coder_B'];Krippendorff's alpha for sentiment: 0.4598
chat_id with no match: [7, 13, 25, 29]
Annotators : ['code_coder_A', 'code_coder_C', 'code_coder_B'];Krippendorff's alpha for critical: 0.4914
chat_id with no match: [2, 4, 7, 10, 19, 22, 29, 30]

Annotators : ['code_coder_A', 'code_coder_B'];Krippendorff's alpha for sentiment: 0.7540
chat_id with no match: [7, 29]
Annotators : ['code_coder_A', 'code_coder_B'];Krippendorff's alpha for critical: 0.5964
chat_id with no match: [7, 10, 22, 29, 30]

Annotators : ['code_coder_B', 'code_coder_C'];Krippendorff's alpha for sentiment: 0.2333
chat_id with no match: [7, 13, 25]
Annotators : ['code_coder_B', 'code_coder_C'];Krippendorff's alpha for critical: 0.3241
chat_id with no match: [2, 4, 7, 10, 19, 29, 30]

Annotators : ['code_coder_A', 'code_coder_C'];Krippendorff's alpha for sentiment: 0.4077
chat_id with no match: [13, 25, 29]
Annotators : ['code_coder_A', 'code_coder_C'];Krippendorff

In [18]:
def cohans_kappa(name: str,
                      annontation = ["code_coder_A", "code_coder_C",  "code_coder_B"],
                        ):
    # task wurde nur in Runde 1 codiert (control_*_task),
    # sentiment/critical in Runde 2 (control_a/b/c)
    if name == "task":
        src_a, src_b, src_c = control_a_task, control_b_task, control_c_task
    else:
        src_a, src_b, src_c = control_a, control_b, control_c
    coder_a = src_a[["chat_id", name]].rename(columns={name: "code_coder_A"})
    coder_b = src_b[["chat_id", name]].rename(columns={name: "code_coder_B"})
    coder_c = src_c[["chat_id", name]].rename(columns={name: "code_coder_C"})
    merged = coder_a.merge(coder_b, on="chat_id").merge(coder_c, on="chat_id")
    kappa = cohen_kappa_score(merged[annontation[0]], merged[annontation[1]])
    return kappa

In [19]:
print(cohans_kappa(name = "task", annontation = coder_AC))
print(cohans_kappa(name = "sentiment", annontation = coder_AC))
print(cohans_kappa(name = "critical", annontation = coder_AC))

0.7969543147208122
0.6538461538461537
0.5348837209302326


In [20]:
df_gs = pd.read_csv(repo / "data/processed/control/gs.csv")
df_gs_merged = df_shuffled.merge(df_gs, on="chat_id", how="right")

In [21]:
df_gs_merged.to_csv(repo / "data/processed/control/gs_merged.csv", index=False)